# Сверка дашборда и портала (final_script_2_1)

Январь–июль 2026. Колонка **ДБ** — витрина актуального скрипта `final_df_period_2026_01_2026_08_final_script_2_1.csv`. Старый `final_script_2` не читается. Колонка **Портал** — месячные Excel `01_Январь_2026.xlsx` … `07_Июль_2026.xlsx`.

Тетрадка не пересобирает витрину и не ходит в озеро. Запуск на Jupyter, каталог `/home/jovyan/documents/Equaring/Data`.

Δ = ДБ − Портал. Знак сумм не меняется.

| Строка | ДБ | Портал |
|---|---|---|
| Всего клиентов | договоры `agr_id` | `ID договора` / `Номер договора` |
| Активные, отчёт (≥1 транзакция) | пусто | договоры отчёта с `Количество операций` > 0 |
| Активные, дашборд (active_terms) | `active_terms` > 0 | те же данные отчёта, Δ = дашборд − отчёт |
| Активные, final_df методом коллеги | наш `final_df`: `trx_cnt` > 0 | те же данные отчёта, Δ = этот расчёт − отчёт |
| Неактивные, отчёт | пусто | остальные договоры отчёта |
| Неактивные, дашборд (active_terms) | остальные договоры дашборда | остальные договоры отчёта |
| Неактивные, final_df методом коллеги | остальные договоры `final_df` | остальные договоры отчёта |
| Приток | месяц `d_valid_from` = месяц `snapshot_month_start` | месяц `Дата регистрации договора` = месяц файла |
| Отток | месяц `d_valid_to` = месяц снимка, дата не пустая | месяц даты закрытия = месяц файла |
| Кол-во ТТ, терминалы, деньги | сумма после свёртки ИНН + договор (`max`, как `build_lake_agg`) | то же |
| Комиссия (% эквайринга) | `SUM(commission_from_ops) / SUM(trx_sum)` | та же доля, в процентах |

Лист `сверка_руб` — деньги в рублях. Лист `сверка_компакт` — оборот в млрд, остальные деньги в млн, счётчики и процент как есть.


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
pd.set_option('display.max_rows', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
MONTHS = [f'2026-{m:02d}' for m in range(1, 8)]
MONTH_LABEL = {m: f'{m[5:7]}.{m[2:4]}' for m in MONTHS}

FINAL_DF_CSV = DATA_DIR / 'final_df_period_2026_01_2026_08_final_script_2_1.csv'
CHECKPOINT_DIR = DATA_DIR / 'checkpoints_final_df_2026_01_2026_08_final_script_2_1'
OUT_XLSX = DATA_DIR / 'portal_vs_dashboard_2026_01_2026_07_final_script_2_1.xlsx'

excel_header = 0
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
    '2026-07': -1,
}
excel_reference_by_month = {
    '2026-01': DATA_DIR / '01_Январь_2026.xlsx',
    '2026-02': DATA_DIR / '02_Февраль_2026.xlsx',
    '2026-03': DATA_DIR / '03_Март_2026.xlsx',
    '2026-04': DATA_DIR / '04_Апрель_2026.xlsx',
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
    '2026-07': DATA_DIR / '07_Июль_2026.xlsx',
}

# Комиссия с операций — раньше общей «Комиссия эквайринга»:
# в col_map final_script это имя стоит и в comm_ops, и в comm_total.
COL_MAP = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
    'term_col': ['Кол-во терминалов', 'Количество терминалов'],
    'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt', 'Кол-во операций'],
    'active_terms_col': [
        'Кол-во активных терминалов', 'Количество активных терминалов',
        'Активные терминалы', 'Активных терминалов', 'active_terms',
    ],
    'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'Сумма транзакций', 'trx_sum'],
    'comm_ops_col': [
        'Комиссия (% с операций)', 'Комиссия \n(% с операций)', 'Комиссия % с операций',
        'Комиссия с операций', 'commission_from_ops',
        'Комиссия эквайринга',
    ],
    'comm_monthly_col': [
        'Комиссия в месяц', 'Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)',
    ],
    'int_component_col': [
        'Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF, руб)',
        'Комиссия МПС (IRF)', 'IRF', 'Комиссия МПС',
    ],
    'chod_col': ['ЧОД', 'chod'],
    'aur_col': ['АУР', 'AUR', 'Aur', 'Аур', 'АУР, руб', 'АУР руб'],
    'amortization_col': [
        'Амортизация', 'Аморт', 'Амортизация терминалов',
        'amortization', 'Amortization', 'Амортизация, руб',
    ],
    'fin_result_col': [
        'Фин. Рез.', 'Фин.Рез.', 'Фин.рез.', 'Фин. рез.',
        'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат',
        'fin_result', 'Fin.Res.', 'FinRes',
    ],
    'open_col': [
        'Дата регистрации договора', 'Дата регистрации',
        'Дата открытия', 'Дата открытия договора', 'Дата начала договора',
        'Дата начала', 'd_valid_from', 'Дата заключения',
    ],
    'close_col': [
        'Дата закрытия', 'Дата закрытия договора', 'Дата расторжения',
        'd_valid_to', 'Дата окончания',
    ],
}

SUM_FIELDS = [
    'retl_cnt', 'term_cnt', 'trx_sum', 'commission_from_ops', 'commission_monthly',
    'int_component', 'chod', 'aur', 'amortization', 'fin_result',
]

METRIC_ROWS = [
    ('clients_total', 'Всего клиентов', 'count'),
    ('clients_active_report', 'Активные, отчёт (≥1 транзакция)', 'count'),
    ('clients_active', 'Активные, дашборд (active_terms)', 'count'),
    ('clients_active_as_report', 'Активные, final_df методом коллеги', 'count'),
    ('clients_inactive_report', 'Неактивные, отчёт', 'count'),
    ('clients_inactive', 'Неактивные, дашборд (active_terms)', 'count'),
    ('clients_inactive_as_report', 'Неактивные, final_df методом коллеги', 'count'),
    ('inflow', 'Приток', 'count'),
    ('outflow', 'Отток', 'count'),
    ('retl_cnt', 'Кол-во ТТ', 'count'),
    ('term_cnt', 'Кол-во терминалов', 'count'),
    ('trx_sum', 'Сумма транзакций', 'bln'),
    ('chod', 'ЧОД', 'mln'),
    ('fin_result', 'Фин. рез.', 'mln'),
    ('aur', 'АУР', 'mln'),
    ('amortization', 'Амортизация', 'mln'),
    ('int_component', 'IRF', 'mln'),
    ('commission_monthly', 'Комиссия месяц', 'mln'),
    ('acq_pct', 'Комиссия (% эквайринга)', 'pct'),
]


def assert_script_21(path):
    if 'final_script_2_1' not in Path(path).name:
        raise RuntimeError(f'Отказ: это не файл final_script_2_1: {path}')


def month_key(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if len(s) >= 7 and s[4] == '-':
        return s[:7]
    try:
        return pd.to_datetime(s).strftime('%Y-%m')
    except Exception:
        return None


def pick_col(columns, candidates):
    cols = list(columns)

    def norm(s):
        s = str(s).replace('\xa0', ' ').replace('\n', ' ').replace('\r', ' ')
        s = re.sub(r'\s+', ' ', s).strip().lower()
        s = re.sub(r'\s*([()])\s*', r'\1', s)
        return s

    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    if s is None:
        return pd.Series(dtype='float64')
    if not isinstance(s, pd.Series):
        s = pd.Series(s)
    raw = s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False)
    raw = raw.str.replace(',', '.', regex=False)
    return pd.to_numeric(raw, errors='coerce')


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None', '<NULL>'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def to_month_label(series):
    """Месяц YYYY-MM. Пустая строка и мусор — пусто. ДД.ММ.ГГГГ и Excel-serial тоже читаются."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return series.dt.strftime('%Y-%m')
    raw = series.astype(str).str.strip()
    raw = raw.replace({
        '': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NULL>': pd.NA, '<NA>': pd.NA,
        'NaT': pd.NA, 'nat': pd.NA, 'NULL': pd.NA,
    })
    try:
        as_dt = pd.to_datetime(raw, errors='coerce', format='mixed', dayfirst=True)
    except (TypeError, ValueError):
        as_dt = pd.to_datetime(raw, errors='coerce', dayfirst=True)
    num = pd.to_numeric(raw, errors='coerce')
    serial = as_dt.isna() & num.notna() & num.between(20000, 80000)
    if serial.any():
        as_dt = as_dt.copy()
        as_dt.loc[serial] = pd.to_datetime(num.loc[serial], unit='D', origin='1899-12-30')
    return as_dt.dt.strftime('%Y-%m')


def first_valid_month(s):
    s = s.dropna()
    s = s[s.astype(str).str.strip().ne('')]
    s = s[s.astype(str) != 'NaT']
    return s.iloc[0] if len(s) else np.nan


def read_month_excel(path, header):
    if header != -1:
        return pd.read_excel(path, header=header)
    raw = pd.read_excel(path, header=None, nrows=12)
    markers = (
        'аур', 'амортизац', 'фин. рез', 'фин.рез', 'фин рез',
        'fin.res', 'fin_result', 'чод', 'инн',
    )
    for i, row in raw.iterrows():
        cells = ' '.join(
            str(x).lower().replace('\n', ' ')
            for x in row.tolist() if pd.notna(x)
        )
        if any(m in cells for m in markers):
            print(f'{path.name}: header=-1 → строка заголовка {i}')
            return pd.read_excel(path, header=i)
    print(f'{path.name}: header=-1, строка заголовка не найдена, header=None')
    return pd.read_excel(path, header=None)


def guess_date_col(columns, kind):
    """kind: open | close. Не берём дату привязки P."""
    if kind == 'open':
        keys = ('открыт', 'дата начала', 'регистрац')
    else:
        keys = ('закрыт', 'расторж')
    hits = []
    for c in columns:
        n = re.sub(r'\s+', ' ', str(c).replace('\n', ' ').strip().lower())
        if 'привяз' in n or n.endswith(' p') or ' p ' in f' {n} ':
            continue
        if any(k in n for k in keys):
            hits.append(c)
    if not hits:
        return None
    for c in hits:
        if 'договор' in str(c).lower():
            return c
    return hits[0]


MONTH_NAME_HINT = {
    '2026-01': 'январ',
    '2026-02': 'феврал',
    '2026-03': 'март',
    '2026-04': 'апрел',
    '2026-05': 'май',
    '2026-06': 'июн',
    '2026-07': 'июл',
}


def resolve_excel_path(month, path):
    """Если канонического имени нет, берём единственный xlsx месяца в DATA_DIR."""
    path = Path(path)
    if path.exists():
        return path
    hint = MONTH_NAME_HINT[month]
    found = []
    if DATA_DIR.exists():
        for p in sorted(DATA_DIR.glob('*.xlsx')):
            name = p.name.lower().replace('ё', 'е')
            if hint in name and '2026' in name:
                found.append(p)
    if len(found) == 1:
        print(f'{month}: {path.name} нет, читаю {found[0].name}')
        return found[0]
    if len(found) > 1:
        print(f'{month}: несколько файлов {[p.name for p in found]}, ожидался {path.name}')
    return path


print('DATA_DIR', DATA_DIR, 'exists=', DATA_DIR.exists())
print('final_df csv exists=', FINAL_DF_CSV.exists(), FINAL_DF_CSV.name)
print('checkpoints exists=', CHECKPOINT_DIR.exists())
for k, v in excel_reference_by_month.items():
    print(f'Excel {k}: exists={Path(v).exists()} | {Path(v).name}')


In [ ]:
def load_final_df():
    if FINAL_DF_CSV.exists():
        assert_script_21(FINAL_DF_CSV)
        df = pd.read_csv(FINAL_DF_CSV, dtype=str, low_memory=False)
        return df, f'csv: {FINAL_DF_CSV.name}'

    if not CHECKPOINT_DIR.exists():
        raise RuntimeError(
            'Нет final_df final_script_2_1. Нужен '
            f'{FINAL_DF_CSV.name} или каталог {CHECKPOINT_DIR.name}. '
            'Старый final_script_2 не подставляется.'
        )
    assert_script_21(CHECKPOINT_DIR)
    month_files = sorted(CHECKPOINT_DIR.glob('final_df_2026_*.parquet')) + sorted(
        CHECKPOINT_DIR.glob('final_df_2026_*.csv')
    )
    if not month_files:
        raise RuntimeError(f'В {CHECKPOINT_DIR} нет final_df_2026_*.parquet/csv')
    frames = []
    seen = set()
    for f in month_files:
        if f.stem in seen:
            continue
        seen.add(f.stem)
        if f.suffix == '.parquet':
            frames.append(pd.read_parquet(f))
        else:
            frames.append(pd.read_csv(f, dtype=str, low_memory=False))
    return pd.concat(frames, ignore_index=True), f'checkpoints: {CHECKPOINT_DIR.name}'


final_df, source_used = load_final_df()
need = {
    'report_month', 'agr_id', 'inn', 'retl_cnt', 'term_cnt', 'active_terms', 'trx_cnt',
    'trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component',
    'chod', 'aur', 'amortization', 'fin_result', 'd_valid_from', 'd_valid_to',
}
missing = need - set(final_df.columns)
if missing:
    raise RuntimeError(f'В final_df нет колонок: {sorted(missing)}')

work = final_df.copy()
work['report_month'] = work['report_month'].map(month_key)
if 'snapshot_month_start' not in work.columns:
    work['snapshot_month_start'] = work['report_month']
    print('snapshot_month_start нет — для притока/оттока берётся report_month')

present = sorted(x for x in work['report_month'].dropna().unique().tolist())
extra = [m for m in present if m not in MONTHS]
work = work[work['report_month'].isin(MONTHS)].copy()
print('source:', source_used)
print('rows Jan–Jul:', f'{len(work):,}', '| months:', sorted(work['report_month'].dropna().unique().tolist()))
if extra:
    print('вне окна, в сверку не входят:', extra)


def dedup_sums(frame, inn_key, agr_key):
    """Одна строка на ИНН + договор, max, затем сумма. Как build_lake_agg."""
    sub = frame.dropna(subset=[agr_key]).copy()
    for c in SUM_FIELDS:
        sub[c] = to_num_series(sub[c])
    if inn_key in sub.columns and sub[inn_key].notna().any():
        sub = sub.dropna(subset=[inn_key])
        keys = [inn_key, agr_key]
    else:
        keys = [agr_key]
    if sub.empty:
        return {c: np.nan for c in SUM_FIELDS}
    agg = sub.groupby(keys, as_index=False)[SUM_FIELDS].max()
    out = {}
    for c in SUM_FIELDS:
        out[c] = float(agg[c].fillna(0).sum())
    return out


def client_and_flow(frame, agr_key, active_col, snap_labels, open_labels, close_labels, active_mode):
    """active_mode: column | trx. snap/open/close — Series месяца YYYY-MM, индекс как у frame."""
    sub = frame.dropna(subset=[agr_key]).copy()
    sub['_snap'] = snap_labels.reindex(sub.index)
    sub['_open'] = open_labels.reindex(sub.index)
    sub['_close'] = close_labels.reindex(sub.index)
    if active_mode == 'trx' and 'trx_cnt' in sub.columns:
        sub['_active'] = to_num_series(sub['trx_cnt'])
    elif active_mode == 'column' and active_col in sub.columns:
        sub['_active'] = to_num_series(sub[active_col])
    else:
        sub['_active'] = to_num_series(sub['term_cnt']) if 'term_cnt' in sub.columns else 0.0
    if sub.empty:
        return {
            'clients_total': np.nan,
            'clients_active': np.nan,
            'clients_inactive': np.nan,
            'inflow': np.nan,
            'outflow': np.nan,
        }
    per = sub.groupby(agr_key, as_index=False).agg(
        _active=('_active', 'max'),
        _snap=('_snap', first_valid_month),
        _open=('_open', first_valid_month),
        _close=('_close', first_valid_month),
    )
    n_total = int(per[agr_key].nunique())
    n_active = int(per.loc[per['_active'].fillna(0) > 0, agr_key].nunique())
    inflow = int(per.loc[per['_open'].notna() & (per['_open'] == per['_snap']), agr_key].nunique())
    outflow = int(per.loc[per['_close'].notna() & (per['_close'] == per['_snap']), agr_key].nunique())
    return {
        'clients_total': n_total,
        'clients_active': n_active,
        'clients_inactive': n_total - n_active,
        'inflow': inflow,
        'outflow': outflow,
    }


def acq_pct(sums):
    trx = sums.get('trx_sum')
    comm = sums.get('commission_from_ops')
    if trx is None or pd.isna(trx) or abs(trx) < 1e-9 or comm is None or pd.isna(comm):
        return np.nan
    return 100.0 * float(comm) / float(trx)


def metrics_from_parts(clients, sums):
    row = dict(clients)
    row.update(sums)
    row['acq_pct'] = acq_pct(sums)
    return row


rows = []
audit = []

for month in MONTHS:
    g = work.loc[work['report_month'] == month].copy()
    g['inn_key'] = g['inn'].map(normalize_inn_q1)
    g['agr_id_key'] = g['agr_id'].map(normalize_agr_q1)
    snap = to_month_label(g['snapshot_month_start'])
    opened = to_month_label(g['d_valid_from'])
    closed = to_month_label(g['d_valid_to'])
    clients_trx = client_and_flow(g, 'agr_id_key', 'trx_cnt', snap, opened, closed, 'trx')
    clients_term = client_and_flow(g, 'agr_id_key', 'active_terms', snap, opened, closed, 'column')
    clients_term['clients_active_as_report'] = clients_trx['clients_active']
    clients_term['clients_inactive_as_report'] = clients_trx['clients_inactive']
    clients_term['clients_active_report'] = np.nan
    clients_term['clients_inactive_report'] = np.nan
    sums = dedup_sums(g, 'inn_key', 'agr_id_key')
    db = metrics_from_parts(clients_term, sums)
    db_status = 'ok' if len(g) else 'no_rows'

    portal = {k: np.nan for k, _label, _scale in METRIC_ROWS}
    path = resolve_excel_path(month, excel_reference_by_month[month])
    if not path.exists():
        print(f'{month}: нет файла портала {path.name}')
        audit.append({'report_month': month, 'field': 'file', 'excel_column': None, 'status': 'file_missing'})
    else:
        header = excel_header_by_month.get(month, excel_header)
        ex = read_month_excel(path, header)
        resolved = {k: pick_col(ex.columns, v) for k, v in COL_MAP.items()}
        if resolved['open_col'] is None:
            resolved['open_col'] = guess_date_col(ex.columns, 'open')
            open_status = 'guessed' if resolved['open_col'] else 'missing'
        else:
            open_status = 'exact'
        if resolved['close_col'] is None:
            resolved['close_col'] = guess_date_col(ex.columns, 'close')
            close_status = 'guessed' if resolved['close_col'] else 'missing'
        else:
            close_status = 'exact'

        if resolved['open_col'] is None or resolved['close_col'] is None:
            print(f'{month}: нет колонки даты для притока/оттока.')
            print('  открытие:', resolved['open_col'], open_status)
            print('  закрытие:', resolved['close_col'], close_status)
            print('  колонки файла:', list(ex.columns))

        if resolved['agr_col'] is None:
            print(f'{month}: нет ID договора. Колонки: {list(ex.columns)}')
            audit.append({'report_month': month, 'field': 'agr', 'excel_column': None, 'status': 'missing'})
        else:
            ex = ex.copy()
            if resolved['inn_col'] is None:
                print(f'{month}: нет ИНН, денежные суммы сворачиваются только по договору')
                ex['inn_key'] = pd.Series(pd.NA, index=ex.index)
            else:
                ex['inn_key'] = ex[resolved['inn_col']].map(normalize_inn_q1)
            ex['agr_id_key'] = ex[resolved['agr_col']].map(normalize_agr_q1)
            for field, col_key in [
                ('retl_cnt', 'retl_col'),
                ('term_cnt', 'term_col'),
                ('trx_sum', 'trx_sum_col'),
                ('commission_from_ops', 'comm_ops_col'),
                ('commission_monthly', 'comm_monthly_col'),
                ('int_component', 'int_component_col'),
                ('chod', 'chod_col'),
                ('aur', 'aur_col'),
                ('amortization', 'amortization_col'),
                ('fin_result', 'fin_result_col'),
            ]:
                col = resolved[col_key]
                ex[field] = to_num_series(ex[col]) if col else np.nan
                audit.append({
                    'report_month': month,
                    'field': field,
                    'excel_column': col,
                    'status': 'ok' if col else 'missing',
                })

            trx_cnt_name = resolved.get('trx_cnt_col')
            if trx_cnt_name:
                ex['trx_cnt'] = to_num_series(ex[trx_cnt_name])
                active_mode = 'trx'
                active_status = 'trx_cnt_gt_0'
            elif resolved.get('trx_sum_col'):
                ex['trx_cnt'] = ex['trx_sum']
                active_mode = 'trx'
                active_status = 'fallback_trx_sum'
                print(f'{month}: нет «Количество операций», активность = Сумма операций > 0')
            else:
                ex['trx_cnt'] = np.nan
                active_mode = 'trx'
                active_status = 'missing'
                print(f'{month}: нет колонки операций, активные клиенты портала пустые')
            audit.append({
                'report_month': month,
                'field': 'active_clients',
                'excel_column': trx_cnt_name,
                'status': active_status,
            })
            audit.append({
                'report_month': month,
                'field': 'open',
                'excel_column': resolved['open_col'],
                'status': open_status,
            })
            audit.append({
                'report_month': month,
                'field': 'close',
                'excel_column': resolved['close_col'],
                'status': close_status,
            })

            file_month = pd.Series(month, index=ex.index)
            open_m = (
                to_month_label(ex[resolved['open_col']])
                if resolved['open_col'] else pd.Series(pd.NA, index=ex.index)
            )
            close_m = (
                to_month_label(ex[resolved['close_col']])
                if resolved['close_col'] else pd.Series(pd.NA, index=ex.index)
            )
            p_clients = client_and_flow(
                ex, 'agr_id_key', 'active_terms', file_month, open_m, close_m, active_mode,
            )
            if active_status == 'missing':
                p_clients['clients_active'] = np.nan
                p_clients['clients_inactive'] = np.nan
            p_clients['clients_active_report'] = p_clients['clients_active']
            p_clients['clients_inactive_report'] = p_clients['clients_inactive']
            p_clients['clients_active_as_report'] = p_clients['clients_active']
            p_clients['clients_inactive_as_report'] = p_clients['clients_inactive']
            if resolved['open_col'] is None:
                p_clients['inflow'] = np.nan
            if resolved['close_col'] is None:
                p_clients['outflow'] = np.nan
            p_sums = dedup_sums(ex, 'inn_key', 'agr_id_key')
            for field, col_key in [
                ('retl_cnt', 'retl_col'),
                ('term_cnt', 'term_col'),
                ('trx_sum', 'trx_sum_col'),
                ('commission_from_ops', 'comm_ops_col'),
                ('commission_monthly', 'comm_monthly_col'),
                ('int_component', 'int_component_col'),
                ('chod', 'chod_col'),
                ('aur', 'aur_col'),
                ('amortization', 'amortization_col'),
                ('fin_result', 'fin_result_col'),
            ]:
                if resolved[col_key] is None:
                    p_sums[field] = np.nan
            portal = metrics_from_parts(p_clients, p_sums)

    for key, _label, _scale in METRIC_ROWS:
        rows.append({
            'report_month': month,
            'metric': key,
            'db': db.get(key, np.nan),
            'portal': portal.get(key, np.nan),
            'db_status': db_status,
        })

long_df = pd.DataFrame(rows)
long_df['delta'] = long_df['db'] - long_df['portal']
audit_df = pd.DataFrame(audit)
print('long rows', len(long_df), 'audit rows', len(audit_df))


In [ ]:
def wide_from(values):
    """values: DataFrame с колонками report_month, metric, db, portal, delta."""
    pieces = []
    for month in MONTHS:
        label = MONTH_LABEL[month]
        part = values.loc[values['report_month'] == month, ['metric', 'db', 'portal', 'delta']].copy()
        part = part.rename(columns={'db': (label, 'ДБ'), 'portal': (label, 'Портал'), 'delta': (label, 'Δ')})
        part = part.set_index('metric')
        pieces.append(part)
    wide = pd.concat(pieces, axis=1)
    wide.columns = pd.MultiIndex.from_tuples(wide.columns)
    order = [k for k, _l, _s in METRIC_ROWS]
    labels = {k: lab for k, lab, _s in METRIC_ROWS}
    wide = wide.reindex(order)
    wide.index = [labels[k] for k in wide.index]
    wide.index.name = 'Показатель'
    return wide


wide_rub = wide_from(long_df)

scale = {k: s for k, _l, s in METRIC_ROWS}
compact_src = long_df.copy()
for key, kind in scale.items():
    mask = compact_src['metric'] == key
    if kind == 'bln':
        compact_src.loc[mask, ['db', 'portal', 'delta']] = compact_src.loc[mask, ['db', 'portal', 'delta']] / 1e9
    elif kind == 'mln':
        compact_src.loc[mask, ['db', 'portal', 'delta']] = compact_src.loc[mask, ['db', 'portal', 'delta']] / 1e6
wide_compact = wide_from(compact_src)

print('Сверка, компактный вид: оборот млрд, остальные деньги млн, % как проценты')
display(wide_compact)
print('Сверка, рубли и штуки')
display(wide_rub)

jan = long_df.loc[long_df['report_month'] == '2026-01'].set_index('metric')
if len(jan):
    total = jan.loc['clients_total', 'db']
    active = jan.loc['clients_active', 'db']
    inactive = jan.loc['clients_inactive', 'db']
    print('Январь ДБ: всего', total, '| активные по терминалу', active, '| неактивные', inactive, '| сумма', active + inactive)
    if pd.notna(total) and abs((active + inactive) - total) > 0.5:
        print('Январь: активные по терминалу + неактивные не равны всему')
    print('Январь, три способа, активные:')
    print('  1 отчёт (≥1 транзакция)', jan.loc['clients_active_report', 'portal'])
    print('  2 дашборд (active_terms)', jan.loc['clients_active', 'db'])
    print('  3 final_df методом коллеги', jan.loc['clients_active_as_report', 'db'])
    print('Январь приток ДБ/Портал', jan.loc['inflow', 'db'], jan.loc['inflow', 'portal'])
    print('Январь отток ДБ/Портал', jan.loc['outflow', 'db'], jan.loc['outflow', 'portal'])
    if pd.isna(jan.loc['inflow', 'portal']) or pd.isna(jan.loc['outflow', 'portal']):
        print('Январь: портальный приток или отток пустой — смотрите лист «колонки»')

pretty = long_df.copy()
pretty['metric'] = pretty['metric'].map({k: lab for k, lab, _s in METRIC_ROWS})
pretty['month'] = pretty['report_month'].map(MONTH_LABEL)

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as writer:
    wide_rub.to_excel(writer, sheet_name='сверка_руб')
    wide_compact.to_excel(writer, sheet_name='сверка_компакт')
    pretty.to_excel(writer, sheet_name='помесячно', index=False)
    if len(audit_df):
        audit_df.to_excel(writer, sheet_name='колонки', index=False)

print('Saved:', OUT_XLSX)


In [ ]:
# Разбор притока и оттока. Запускать после ячейки сверки.
# Ищет месяцы, где |ДБ − Портал| > 2, и для каждого договора пишет,
# чем дата в final_df отличается от даты в Excel.
# Отдельно считает портал с dayfirst=True: если февраль сходится только так,
# в файле даты вида ДД.ММ.ГГГГ прочитались как месяц.день.

FLOW_GAP = 2
FLOW_XLSX = DATA_DIR / 'portal_vs_dashboard_flow_gap_2026_01_2026_07.xlsx'


def _blank_to_na(series):
    raw = series.astype(str).str.strip()
    return raw.replace({
        '': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NULL>': pd.NA, '<NA>': pd.NA,
        'NaT': pd.NA, 'nat': pd.NA, 'NULL': pd.NA,
    })


def month_label_dayfirst(series):
    """То же, что to_month_label, но неоднозначные даты читаются как ДД.ММ.ГГГГ."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return series.dt.strftime('%Y-%m')
    raw = _blank_to_na(series)
    try:
        as_dt = pd.to_datetime(raw, errors='coerce', format='mixed', dayfirst=True)
    except (TypeError, ValueError):
        as_dt = pd.to_datetime(raw, errors='coerce', dayfirst=True)
    num = pd.to_numeric(raw, errors='coerce')
    serial = as_dt.isna() & num.notna() & num.between(20000, 80000)
    if serial.any():
        as_dt = as_dt.copy()
        as_dt.loc[serial] = pd.to_datetime(num.loc[serial], unit='D', origin='1899-12-30')
    return as_dt.dt.strftime('%Y-%m')


def _first_raw(s):
    s = s.dropna()
    s = s[s.astype(str).str.strip().ne('')]
    s = s[s.astype(str) != 'NaT']
    return s.iloc[0] if len(s) else np.nan


def flow_frame(frame, agr_col, snap, opened, closed, raw_open=None, raw_close=None):
    sub = frame.dropna(subset=[agr_col]).copy()
    sub['_snap'] = snap.reindex(sub.index)
    sub['_open'] = opened.reindex(sub.index)
    sub['_close'] = closed.reindex(sub.index)
    if raw_open is not None:
        sub['_raw_open'] = raw_open.reindex(sub.index)
    else:
        sub['_raw_open'] = sub['_open']
    if raw_close is not None:
        sub['_raw_close'] = raw_close.reindex(sub.index)
    else:
        sub['_raw_close'] = sub['_close']
    if sub.empty:
        return pd.DataFrame(columns=[agr_col, '_snap', '_open', '_close', '_raw_open', '_raw_close'])
    return sub.groupby(agr_col, as_index=False).agg(
        _snap=('_snap', first_valid_month),
        _open=('_open', first_valid_month),
        _close=('_close', first_valid_month),
        _raw_open=('_raw_open', _first_raw),
        _raw_close=('_raw_close', _first_raw),
    )


def _ids(per, agr_col, kind):
    col = '_open' if kind == 'inflow' else '_close'
    hit = per[col].notna() & (per[col] == per['_snap'])
    return set(per.loc[hit, agr_col].astype(str))


def _why(row, kind, file_month):
    """Почему договор есть только на одной стороне."""
    side = row['side']
    raw = row['excel_raw_open'] if kind == 'inflow' else row['excel_raw_close']
    parsed = row['excel_open_month'] if kind == 'inflow' else row['excel_close_month']
    lake = row['lake_open_month'] if kind == 'inflow' else row['lake_close_month']
    raw_s = '' if pd.isna(raw) else str(raw).strip()
    if side == 'only_dashboard':
        if row['in_excel'] != 1:
            return 'договора нет в Excel'
        if raw_s in ('', 'nan', 'None', 'NaT'):
            return 'в Excel дата пустая'
        if pd.isna(parsed) or str(parsed) in ('', 'NaT', 'nan'):
            return 'дату Excel не разобрали'
        if str(parsed) != file_month:
            return f'в Excel месяц {parsed}, не {file_month}'
        return 'дата Excel в этом месяце, но договор не попал в приток/отток портала'
    if row['in_lake'] != 1:
        return 'договора нет в final_df'
    if pd.isna(lake) or str(lake) in ('', 'NaT', 'nan'):
        return 'в final_df дата пустая'
    if str(lake) != file_month:
        return f'в final_df месяц {lake}, не {file_month}'
    return 'дата final_df в этом месяце, но в сверку портала договор не вошёл'


gap_months = []
for month in MONTHS:
    part = long_df.loc[long_df['report_month'] == month].set_index('metric')
    inn_d = abs(float(part.loc['inflow', 'delta'])) if pd.notna(part.loc['inflow', 'delta']) else 0
    out_d = abs(float(part.loc['outflow', 'delta'])) if pd.notna(part.loc['outflow', 'delta']) else 0
    if inn_d > FLOW_GAP or out_d > FLOW_GAP or month == '2026-02':
        gap_months.append(month)
print('Месяцы разбора:', gap_months)

parser_rows = []
detail_frames = []

for month in MONTHS:
    g = work.loc[work['report_month'] == month].copy()
    g['agr_id_key'] = g['agr_id'].map(normalize_agr_q1)
    snap = to_month_label(g['snapshot_month_start'])
    lake_open = to_month_label(g['d_valid_from'])
    lake_close = to_month_label(g['d_valid_to'])
    lake = flow_frame(g, 'agr_id_key', snap, lake_open, lake_close, g['d_valid_from'], g['d_valid_to'])
    lake_in = _ids(lake, 'agr_id_key', 'inflow')
    lake_out = _ids(lake, 'agr_id_key', 'outflow')

    path = resolve_excel_path(month, excel_reference_by_month[month])
    portal_in = set()
    portal_out = set()
    portal_in_dfirst = set()
    portal_out_dfirst = set()
    excel_per = pd.DataFrame(columns=['agr_id_key', '_open', '_close', '_raw_open', '_raw_close', '_open_dfirst', '_close_dfirst'])
    open_name = close_name = None
    if path.exists():
        header = excel_header_by_month.get(month, excel_header)
        ex = read_month_excel(path, header)
        resolved = {k: pick_col(ex.columns, v) for k, v in COL_MAP.items()}
        if resolved['open_col'] is None:
            resolved['open_col'] = guess_date_col(ex.columns, 'open')
        if resolved['close_col'] is None:
            resolved['close_col'] = guess_date_col(ex.columns, 'close')
        open_name = resolved['open_col']
        close_name = resolved['close_col']
        if resolved['agr_col'] is not None and open_name and close_name:
            ex = ex.copy()
            ex['agr_id_key'] = ex[resolved['agr_col']].map(normalize_agr_q1)
            file_snap = pd.Series(month, index=ex.index)
            ex_open = to_month_label(ex[open_name])
            ex_close = to_month_label(ex[close_name])
            ex_open_d = month_label_dayfirst(ex[open_name])
            ex_close_d = month_label_dayfirst(ex[close_name])
            excel_per = flow_frame(
                ex, 'agr_id_key', file_snap, ex_open, ex_close, ex[open_name], ex[close_name],
            )
            excel_d = flow_frame(
                ex, 'agr_id_key', file_snap, ex_open_d, ex_close_d, ex[open_name], ex[close_name],
            )
            excel_per = excel_per.merge(
                excel_d[['agr_id_key', '_open', '_close']].rename(
                    columns={'_open': '_open_dfirst', '_close': '_close_dfirst'}
                ),
                on='agr_id_key',
                how='left',
            )
            portal_in = _ids(excel_per, 'agr_id_key', 'inflow')
            portal_out = _ids(excel_per, 'agr_id_key', 'outflow')
            portal_in_dfirst = _ids(
                excel_d.rename(columns={'_open': '_open', '_close': '_close'}),
                'agr_id_key',
                'inflow',
            )
            # _ids uses _open/_close; excel_d already has those as dayfirst months
            portal_in_dfirst = _ids(excel_d, 'agr_id_key', 'inflow')
            portal_out_dfirst = _ids(excel_d, 'agr_id_key', 'outflow')

    parser_rows.append({
        'report_month': month,
        'inflow_dashboard': len(lake_in),
        'inflow_portal': len(portal_in),
        'inflow_portal_dayfirst': len(portal_in_dfirst),
        'outflow_dashboard': len(lake_out),
        'outflow_portal': len(portal_out),
        'outflow_portal_dayfirst': len(portal_out_dfirst),
        'excel_open_col': open_name,
        'excel_close_col': close_name,
    })

    if month not in gap_months:
        continue

    lake_k = lake.rename(columns={
        'agr_id_key': 'agr_id',
        '_open': 'lake_open_month',
        '_close': 'lake_close_month',
        '_raw_open': 'lake_raw_open',
        '_raw_close': 'lake_raw_close',
    })
    ex_k = excel_per.rename(columns={
        'agr_id_key': 'agr_id',
        '_open': 'excel_open_month',
        '_close': 'excel_close_month',
        '_raw_open': 'excel_raw_open',
        '_raw_close': 'excel_raw_close',
        '_open_dfirst': 'excel_open_month_dayfirst',
        '_close_dfirst': 'excel_close_month_dayfirst',
    })
    both = lake_k.merge(ex_k, on='agr_id', how='outer', indicator=True)
    both['in_lake'] = (both['_merge'] != 'right_only').astype(int)
    both['in_excel'] = (both['_merge'] != 'left_only').astype(int)
    both = both.drop(columns=['_merge', '_snap_x', '_snap_y'], errors='ignore')

    for kind, lake_ids, portal_ids in (
        ('приток', lake_in, portal_in),
        ('отток', lake_out, portal_out),
    ):
        only_db = lake_ids - portal_ids
        only_xl = portal_ids - lake_ids
        print(
            f'{month} {kind}: только дашборд {len(only_db)}, только отчёт {len(only_xl)}'
        )
        gap_ids = only_db | only_xl
        if not gap_ids:
            continue
        gap = both.loc[both['agr_id'].astype(str).isin(gap_ids)].copy()
        gap['side'] = np.where(
            gap['agr_id'].astype(str).isin(only_db),
            'only_dashboard',
            'only_report',
        )
        gap['kind'] = kind
        gap['report_month'] = month
        gap['why'] = gap.apply(lambda r: _why(r, 'inflow' if kind == 'приток' else 'outflow', month), axis=1)
        detail_frames.append(gap)
        print(gap['why'].value_counts().to_string())
        print()

parser_df = pd.DataFrame(parser_rows)
print('Сравнение разбора дат. dayfirst — чтение ДД.ММ.ГГГГ.')
display(parser_df)

if detail_frames:
    flow_gap_df = pd.concat(detail_frames, ignore_index=True)
    keep = [
        'report_month', 'kind', 'side', 'why', 'agr_id',
        'lake_open_month', 'excel_open_month', 'excel_open_month_dayfirst', 'lake_raw_open', 'excel_raw_open',
        'lake_close_month', 'excel_close_month', 'excel_close_month_dayfirst', 'lake_raw_close', 'excel_raw_close',
        'in_lake', 'in_excel',
    ]
    keep = [c for c in keep if c in flow_gap_df.columns]
    flow_gap_df = flow_gap_df[keep]
    with pd.ExcelWriter(FLOW_XLSX, engine='openpyxl') as writer:
        parser_df.to_excel(writer, sheet_name='сводка_дат', index=False)
        flow_gap_df.to_excel(writer, sheet_name='договоры', index=False)
    print('Список договоров:', FLOW_XLSX, 'строк', len(flow_gap_df))
else:
    print('Расхождений больше', FLOW_GAP, 'нет')


In [ ]:
# Почему договор отчёта нет в final_df.
# Сначала ищет agr_id во всех месяцах CSV. Если в снимке месяца его нет,
# смотрит озеро тем же периметром, что секция 01 final_script_2_1:
#   SA, не удалён, компания с ИНН,
#   d_valid_from <= конец месяца, d_valid_to пусто или >= начала месяца,
#   есть терминал cf_ter_type = P, у которого d_valid_to пусто или > начала месяца.
# Запускать после ячейки сверки. Список берётся из предыдущей ячейки или из xlsx.

CHECK_MONTH = '2026-07'
LOOKUP_LAKE = True
JULY_EXAMPLES = [
    '260656662093',
    '276769238491',
    '276769238775',
    '276769238777',
    '641941765557',
    '740831861115',
    '991536343927',
]
MISSING_XLSX = DATA_DIR / 'portal_vs_dashboard_missing_final_df_2026_07.xlsx'


def _ensure_imp():
    if globals().get('imp') is not None:
        return globals()['imp']
    if 'connect' not in globals():
        return None
    conn = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    conn._init_connection()
    return conn


def _impala_fetch(conn, sql):
    with conn:
        conn.execute('set MEM_LIMIT=4g')
        return conn.fetch(sql)


def _sql_in(ids):
    clean = []
    for raw in ids:
        s = str(raw).strip()
        if not s.isdigit():
            raise ValueError(f'нечисловой agr_id: {raw}')
        clean.append("'" + s + "'")
    return ','.join(clean)


def _as_ts(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return pd.NaT
    text = str(value).strip()
    if text in ('', 'nan', 'None', 'NaT', 'NULL'):
        return pd.NaT
    return pd.to_datetime(text, errors='coerce')


def _deleted(flag):
    """В скрипте отсекается только ods_deleted_flg = 1."""
    return str(flag).strip() == '1'


def _covers_agreement(row, month_start, month_end):
    opened = _as_ts(row['d_valid_from'])
    closed = _as_ts(row['d_valid_to'])
    if pd.isna(opened) or opened > month_end:
        return False
    if pd.notna(closed) and closed < month_start:
        return False
    return True


def _covers_term(row, month_start, month_end):
    """Как в скрипте: d_valid_to пусто или строго позже начала месяца."""
    opened = _as_ts(row['term_from'])
    closed = _as_ts(row['term_to'])
    if pd.isna(opened) or opened > month_end:
        return False
    if pd.notna(closed) and closed <= month_start:
        return False
    return True


def _span(rows):
    parts = []
    view = rows.copy()
    view['_from'] = view['d_valid_from'].map(_as_ts)
    view = view.sort_values('_from')
    for _, row in view.iterrows():
        closed = row['d_valid_to']
        closed = 'открыт' if pd.isna(closed) or str(closed).strip() in ('', 'None', 'NaT', 'nan') else closed
        parts.append(
            f"{row['acq_class'] or '?'} {row['d_valid_from']}..{closed} del={row['agr_deleted']}"
        )
    return ' | '.join(parts[:8])


if 'flow_gap_df' in globals():
    gap = flow_gap_df.copy()
else:
    gap = pd.read_excel(FLOW_XLSX, sheet_name='договоры', dtype=str)

gap['agr_id'] = gap['agr_id'].map(normalize_agr_q1)
examples = {normalize_agr_q1(x) for x in JULY_EXAMPLES}
missing = gap[
    (gap['report_month'] == CHECK_MONTH) & (gap['why'] == 'договора нет в final_df')
].copy()
example_only = sorted(examples - set(missing['agr_id'].dropna()))
if example_only:
    missing = pd.concat([
        missing,
        pd.DataFrame({
            'report_month': CHECK_MONTH,
            'kind': '',
            'agr_id': example_only,
            'why': 'договора нет в final_df',
            'excel_raw_close': pd.NA,
            'excel_raw_open': pd.NA,
        }),
    ], ignore_index=True)

check_ids = sorted(x for x in set(missing['agr_id'].dropna()) | examples if x)
print(f'{CHECK_MONTH}: договоров отчёта без final_df — {len(check_ids)}')

lake_all = final_df.copy()
lake_all['agr_id_key'] = lake_all['agr_id'].map(normalize_agr_q1)
lake_all['report_month'] = lake_all['report_month'].map(month_key)
lake_all['open_m'] = to_month_label(lake_all['d_valid_from'])
lake_all['close_m'] = to_month_label(lake_all['d_valid_to'])

rows = []
for agr_id in check_ids:
    hit = lake_all.loc[lake_all['agr_id_key'] == agr_id]
    months = sorted(x for x in hit['report_month'].dropna().unique().tolist())
    gap_rows = missing.loc[missing['agr_id'] == agr_id]
    kinds = ', '.join(sorted(x for x in gap_rows['kind'].dropna().astype(str).unique() if x))
    excel_close = gap_rows['excel_raw_close'].dropna().astype(str)
    excel_close = excel_close.iloc[0] if len(excel_close) else ''
    excel_open = gap_rows['excel_raw_open'].dropna().astype(str)
    excel_open = excel_open.iloc[0] if len(excel_open) else ''
    if CHECK_MONTH in months:
        local = f'есть в final_df за {CHECK_MONTH}'
    elif months:
        one = hit.sort_values('report_month').iloc[-1]
        local = (
            f"в final_df есть в {', '.join(months)}, в {CHECK_MONTH} нет; "
            f"последняя строка {one['report_month']}: "
            f"открытие {one['d_valid_from']}, закрытие {one['d_valid_to']}"
        )
    else:
        local = 'нет ни в одном месяце final_df'
    rows.append({
        'agr_id': agr_id,
        'example': int(agr_id in examples),
        'kind': kinds,
        'excel_open': excel_open,
        'excel_close': excel_close,
        'months_in_final_df': ', '.join(months),
        'local': local,
    })

local_df = pd.DataFrame(rows)
month_start = pd.Timestamp(CHECK_MONTH + '-01')
month_end = (month_start + pd.offsets.MonthEnd(0)).normalize()
need_lake = local_df.loc[~local_df['months_in_final_df'].fillna('').str.contains(CHECK_MONTH), 'agr_id'].tolist()

agr_df = pd.DataFrame()
term_df = pd.DataFrame()
lake_error = ''
if LOOKUP_LAKE and need_lake:
    conn = _ensure_imp()
    if conn is None:
        lake_error = 'нет соединения imp и функции connect — озеро не смотрел'
        print(lake_error)
    else:
        print(f'Смотрю в озере {len(need_lake)} договоров')
        ids_sql = _sql_in(need_lake)
        where_ids = f"""
            cast(a.abs_agr_id as string) in ({ids_sql})
            or cast(a.n_agr as string) in ({ids_sql})
            or cast(a.c_agr_number as string) in ({ids_sql})
            or regexp_replace(cast(a.c_agr_number as string), '[^0-9]', '') in ({ids_sql})
        """
        sql_agr = f"""
        select
          cast(a.abs_agr_id as string) as abs_agr_id,
          cast(a.n_agr as string) as n_agr,
          cast(a.c_agr_number as string) as c_agr_number,
          upper(trim(cast(a.acq_class as string))) as acq_class,
          cast(a.d_valid_from as string) as d_valid_from,
          cast(a.d_valid_to as string) as d_valid_to,
          coalesce(cast(a.ods_deleted_flg as string), '0') as agr_deleted,
          case when c.n_cmp is null then '0' else '1' end as has_company,
          coalesce(cast(c.ods_deleted_flg as string), '0') as cmp_deleted,
          case
            when c.c_inn is null or trim(cast(c.c_inn as string)) = '' then '0'
            else '1'
          end as has_inn
        from ods_alpha.scd1_agreements a
        left join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where {where_ids}
        """
        sql_term = f"""
        select
          cast(a.abs_agr_id as string) as abs_agr_id,
          cast(a.n_agr as string) as n_agr,
          upper(trim(cast(t.cf_ter_type as string))) as cf_ter_type,
          cast(t.d_valid_from as string) as term_from,
          cast(t.d_valid_to as string) as term_to,
          coalesce(cast(t.ods_deleted_flg as string), '0') as term_deleted
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_agr_terms t
          on cast(t.n_agr as string) = cast(a.n_agr as string)
        where {where_ids}
        """
        try:
            agr_df = _impala_fetch(conn, sql_agr)
            term_df = _impala_fetch(conn, sql_term)
            print(f'озеро: версий договора {len(agr_df)}, строк терминалов {len(term_df)}')
        except Exception as exc:
            lake_error = f'{type(exc).__name__}: {exc}'
            print('Запрос к озеру не выполнился:', lake_error)

if len(agr_df):
    for col in ('abs_agr_id', 'n_agr', 'c_agr_number'):
        agr_df[col + '_key'] = agr_df[col].map(normalize_agr_q1)
    agr_df['c_agr_number_key'] = agr_df['c_agr_number'].map(
        lambda v: normalize_agr_q1(re.sub(r'\D', '', str(v)) if pd.notna(v) else v)
    )
if len(term_df):
    term_df['abs_agr_id_key'] = term_df['abs_agr_id'].map(normalize_agr_q1)
    term_df['n_agr_key'] = term_df['n_agr'].map(normalize_agr_q1)


def _lake_reason(agr_id):
    if lake_error:
        return lake_error
    if not len(agr_df):
        return 'озеро не запрашивалось'
    by_abs = agr_df.loc[agr_df['abs_agr_id_key'] == agr_id]
    if by_abs.empty:
        other = agr_df.loc[
            (agr_df['n_agr_key'] == agr_id) | (agr_df['c_agr_number_key'] == agr_id)
        ]
        if other.empty:
            return 'нет в ods_alpha.scd1_agreements ни как abs_agr_id, ни как n_agr, ни как номер договора'
        sample = other.iloc[0]
        return (
            'в отчёте не abs_agr_id: найден как n_agr или номер. '
            f"abs_agr_id={sample['abs_agr_id']}, n_agr={sample['n_agr']}, "
            f"номер={sample['c_agr_number']}, класс={sample['acq_class']}"
        )
    alive = by_abs.loc[~by_abs['agr_deleted'].map(_deleted)]
    if alive.empty:
        return 'все версии договора удалены (ods_deleted_flg). ' + _span(by_abs)
    sa = alive.loc[alive['acq_class'].astype(str).str.upper() == 'SA']
    if sa.empty:
        classes = sorted(x for x in alive['acq_class'].dropna().astype(str).unique())
        return f"класс не SA: {', '.join(classes)}. " + _span(alive)
    covering = sa.loc[sa.apply(lambda r: _covers_agreement(r, month_start, month_end), axis=1)]
    if covering.empty:
        return f'окно SA не покрывает {CHECK_MONTH}. ' + _span(sa)
    good = covering.loc[
        (covering['has_company'] == '1')
        & ~covering['cmp_deleted'].map(_deleted)
        & (covering['has_inn'] == '1')
    ]
    if good.empty:
        return 'договор SA на месяц есть, компания удалена, без ИНН или без связи'
    n_agrs = set(good['n_agr_key'].dropna())
    terms = term_df.loc[term_df['n_agr_key'].isin(n_agrs)] if len(term_df) else term_df
    p_terms = terms.loc[
        (terms['cf_ter_type'] == 'P') & ~terms['term_deleted'].map(_deleted)
    ] if len(terms) else terms
    ok_terms = p_terms.loc[
        p_terms.apply(lambda r: _covers_term(r, month_start, month_end), axis=1)
    ] if len(p_terms) else p_terms
    if not len(ok_terms):
        if not len(p_terms):
            return 'на договоре нет терминала типа P'
        closed = ', '.join(
            sorted(x for x in p_terms['term_to'].dropna().astype(str).unique())[:8]
        )
        return (
            f'терминал P есть, но не действует после {month_start.date()} '
            f'(в скрипте условие d_valid_to > начала месяца). даты закрытия терминалов: {closed}'
        )
    return 'фильтры периметра месяца проходят, а в final_df договора нет — смотреть сборку снимка'


reasons = []
lake_spans = []
for agr_id, months in zip(local_df['agr_id'], local_df['months_in_final_df']):
    if CHECK_MONTH in str(months).split(', '):
        reasons.append('есть в снимке этого месяца')
        lake_spans.append('')
        continue
    if not LOOKUP_LAKE:
        reasons.append(local_df.loc[local_df['agr_id'] == agr_id, 'local'].iloc[0])
        lake_spans.append('')
        continue
    reasons.append(_lake_reason(agr_id))
    if len(agr_df):
        hit = agr_df.loc[agr_df['abs_agr_id_key'] == agr_id]
        lake_spans.append(_span(hit) if len(hit) else '')
    else:
        lake_spans.append('')

local_df['lake_versions'] = lake_spans
local_df['reason'] = reasons
local_df = local_df.sort_values(['example', 'kind', 'agr_id'], ascending=[False, True, True])

print()
print('Причины:')
print(local_df['reason'].value_counts().to_string())
print()
print('Примеры июля:')
show_cols = [
    'agr_id', 'kind', 'excel_open', 'excel_close',
    'months_in_final_df', 'lake_versions', 'reason',
]
display(local_df.loc[local_df['example'] == 1, show_cols])
print('Все договора июля, которых нет в final_df:')
display(local_df[show_cols])

local_df.to_excel(MISSING_XLSX, index=False)
print('Saved:', MISSING_XLSX)
